In [ ]:
# Install necessary libraries
%pip install -Uqq langchain-weaviate langchain langchain_mistralai langchain-community beautifulsoup4 weaviate-client sentence-transformers transformers

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from bs4 import BeautifulSoup
import weaviate
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import RecursiveUrlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer
from langchain_weaviate.vectorstores import WeaviateVectorStore
from langchain_mistralai import ChatMistralAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate


In [ ]:
os.environ["MISTRAL_API_KEY"] = "lGSUJ1oM7Eav2Kl7YAYqr6rxFNzNZkXf"

# Paste your credentials here
WEAVIATE_URL = "y927gj2pqscvxn7yy098w.c0.europe-west3.gcp.weaviate.cloud"  # Example: https://someproject.weaviate.network
WEAVIATE_API_KEY = "PfhvUSPPn5mJSd6RtAbzCKY3RFPMmN4lILxe"

## 1. Scrapping des informations

In [ ]:
def bs4_extractor(html: str) -> str:
    soup = BeautifulSoup(html, 'html.parser')
    title = soup.title.string if soup.title else "No title"
    paragraphs = [p.get_text() for p in soup.find_all('p')]
    return title + "\n" + "\n".join(paragraphs)

url_to_scrape = "https://starwars.fandom.com/wiki/Jedha"  # Change this if you want another page

loader = RecursiveUrlLoader(
    url_to_scrape,
    max_depth=1,
    use_async=False,
    extractor=bs4_extractor,
    timeout=10,
    continue_on_failure=True,
    prevent_outside=True
)

docs = loader.load()
print("Loaded", len(docs), "document(s). Example content:\n", docs[0].page_content[:500])


Loaded 1 document(s). Example content:
 Jedha | Wookieepedia | Fandom
Wookieepedia
To remove ads, create an account.Join Wookieepedia today!

READ MORE


Content approaching.

Tales of Enlightenment: New Prospects, The High Republic: Convergence, Tales of Enlightenment: A Different Perspective, The High Republic Adventures (2022) 4, Peace and Unity, Star Wars: The High Republic (Marvel Comics 2022), The High Republic: The Battle of Jedha, The High Republic Adventures (2022) 5, The High Republic Adventures (2022) 6, The High Republic A


## 2. Preparation des documents (tokenization + split)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-cased")
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tokenizer)

splitted_docs = splitter.split_documents(docs)

print("Initial docs:", len(docs))
print("Splitted docs:", len(splitted_docs))


/opt/homebrew/Caskroom/miniconda/base/envs/langchain-client/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initial docs: 1
Splitted docs: 4


# 3. Connection à la base de données 

In [ ]:
client = weaviate.connect_to_wcs(
    cluster_url=WEAVIATE_URL,
    auth_credentials=weaviate.auth.AuthApiKey(WEAVIATE_API_KEY),
)

print("Weaviate Ready?", client.is_ready())


/var/folders/_p/stbgbx154_v2mk71nn05yw5r0000gn/T/ipykernel_629/2510088804.py:7: DeprecatedWarning: connect_to_wcs is deprecated as of 4.6.2. 
This method is deprecated and will be removed in a future release. Use :func:`connect_to_weaviate_cloud` instead.

  client = weaviate.connect_to_wcs(
/opt/homebrew/Caskroom/miniconda/base/envs/langchain-client/lib/python3.10/site-packages/deprecation.py:260: DeprecationWarning: This method is deprecated and will be removed in a future release. Use `connect_to_weaviate_cloud` instead.
  return function(*args, **kwargs)


Weaviate Ready? True


# 4. Envoyer les informations à la BDD

In [ ]:
embeddings = HuggingFaceEmbeddings()

tenant_name = "Wookieepedia"  # Change if needed # equivalent du nom de la BDD

vectorstore = WeaviateVectorStore.from_documents(
    splitted_docs,
    embeddings,
    client=client,
    by_text=False,
    tenant=tenant_name  # Required when multi-tenancy is enabled
)

print("Documents uploaded to Weaviate under tenant:", tenant_name)

2025-May-17 12:43 PM - langchain_weaviate.vectorstores - INFO - Tenant Wookieepedia does not exist in index LangChain_50be51bfc8f646138c8fd9f785a0d5e1. Creating tenant.


Vector Store Created ✅


In [ ]:
query = "What was the original name of Jedha?"
docs = vectorstore.similarity_search(query, k=2, tenant="Wookieepedia")

for idx, doc in enumerate(docs, 1):
    print(f"--- Document {idx} ---\n{doc.page_content[:500]}\n")


# 5. Quetionner le systeme LLM+RAG

In [ ]:
client = weaviate.connect_to_wcs(
    cluster_url=WEAVIATE_URL,
    auth_credentials=weaviate.auth.AuthApiKey(WEAVIATE_API_KEY),
)

# Check connection
print("Weaviate Ready?", client.is_ready())

/var/folders/_p/stbgbx154_v2mk71nn05yw5r0000gn/T/ipykernel_629/2497538572.py:7: DeprecatedWarning: connect_to_wcs is deprecated as of 4.6.2. 
This method is deprecated and will be removed in a future release. Use :func:`connect_to_weaviate_cloud` instead.

  client = weaviate.connect_to_wcs(
/opt/homebrew/Caskroom/miniconda/base/envs/langchain-client/lib/python3.10/site-packages/deprecation.py:260: DeprecationWarning: This method is deprecated and will be removed in a future release. Use `connect_to_weaviate_cloud` instead.
  return function(*args, **kwargs)


Weaviate Ready? True


In [ ]:
llm = ChatMistralAI(model="mistral-large-latest")

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2, "tenant": tenant_name}  # Specify tenant here as well
)

prompt_template = """
You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

Question: {question}

Context: {context}

Answer:
"""

prompt = ChatPromptTemplate(("system", prompt_template))

def format_docs(docs): # fct helper 
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser() # parser : rendre plus lisible le resultat 
)


In [ ]:
response = rag_chain.invoke("What was the initial name of Jedha?")
print("Answer:", response)

Answer: The initial name of Jedha was NiJedha. This name was particularly associated with the Holy City located on the moon. The moon Jedha itself has also been known by other names such as the Pilgrim Moon, the Cold Moon, or the Kyber Heart.
